# Affinity-Guided LIVECell Training

Clone the repository, install dependencies, train on the attached LIVECell dataset, and copy the run artifacts to Kaggle output.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPOSITORY_URL = "https://github.com/EugenTheMachine/some_model.git"
REPOSITORY_DIR = Path("/kaggle/working/some_model_repo")
DATASET_SOURCE = "eugenfromkharkov/livecell-cellseg1-a172"
DATASET_SLUG = DATASET_SOURCE.split("/", 1)[1]
DATASET_ROOT = Path("/kaggle/input") / DATASET_SLUG

if REPOSITORY_DIR.exists():
    shutil.rmtree(REPOSITORY_DIR)
subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(REPOSITORY_DIR)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", str(REPOSITORY_DIR / "requirements.txt")], check=True)
sys.path.insert(0, str(REPOSITORY_DIR / "src"))

import torch

if not torch.cuda.is_available():
    raise RuntimeError("A Kaggle GPU is required.")
GPU_NAME = torch.cuda.get_device_name(0)
if "T4" not in GPU_NAME.upper():
    raise RuntimeError(f"Expected a Tesla T4 GPU, found {GPU_NAME!r}.")
if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(f"Dataset is not mounted at {DATASET_ROOT}.")
print(f"Using {GPU_NAME}; dataset: {DATASET_ROOT}")

In [ ]:
from model import InstanSegModel

epochs = int(os.environ.get("KAGGLE_TRAIN_EPOCHS", "20"))
imgsz = int(os.environ.get("KAGGLE_TRAIN_IMGSZ", "512"))
train_data_ratio = float(os.environ.get("KAGGLE_TRAIN_DATA_RATIO", "1.0"))
if imgsz <= 0 or imgsz % 16:
    raise ValueError("KAGGLE_TRAIN_IMGSZ must be a positive multiple of 16.")

serialized_datasets = list(DATASET_ROOT.rglob("*_dataset.pth"))
if not serialized_datasets:
    serialized_datasets = list(DATASET_ROOT.rglob("*.pth"))
if len(serialized_datasets) > 1:
    raise ValueError(f"Expected one serialized dataset, found {serialized_datasets}")

# trainer = InstanSegModel(
#     data_dir=str(DATASET_ROOT if serialized_datasets else DATASET_ROOT),
#     dataset=str(serialized_datasets[0]) if serialized_datasets else None,
#     experiment_name="kaggle_livecell_ag_mlp",
#     model_name="affinity_guided_mlp",
#     imgsz=imgsz,
#     train_data_ratio=train_data_ratio,
#     seed=42,
#     dim_in=1,
#     epochs=epochs,
#     patience=2,
#     batch_size=2,
#     val_ratio=0.2,
#     test_ratio=0.2,
#     val_interval=1,
#     warmup_epochs=0,
#     resume=False,
#     amp=True,
#     num_workers=0,
#     input_channels=1,
#     backbone_channels=[16, 32, 64, 128, 256],
#     feature_channels=16,
#     num_slots=800,
#     mlp_channels=[32, 16],
#     use_aggregation=True,
# )
# history = trainer.train()
# print(f"Completed {len(history)} epoch(s).")

In [ ]:
# # DELA-Net: foreground + four affinities + normalized embeddings + learned slots.
# dela_num_slots = int(os.environ.get("KAGGLE_TRAIN_DELA_SLOTS", "800"))
# if dela_num_slots <= 0:
#     raise ValueError("KAGGLE_TRAIN_DELA_SLOTS must be positive.")

# dela_trainer = InstanSegModel(
#     data_dir=str(DATASET_ROOT),
#     dataset=str(serialized_datasets[0]) if serialized_datasets else None,
#     experiment_name="kaggle_livecell_dela",
#     model_name="dela_net",
#     imgsz=imgsz,
#     train_data_ratio=train_data_ratio,
#     seed=42,
#     dim_in=1,
#     epochs=epochs,
#     patience=2,
#     batch_size=1,
#     val_ratio=0.2,
#     test_ratio=0.2,
#     val_interval=1,
#     warmup_epochs=0,
#     resume=False,
#     amp=True,
#     num_workers=0,
#     input_channels=1,
#     backbone_channels=[16, 32, 64, 128, 256],
#     num_slots=dela_num_slots,
#     embedding_dim=8,
#     assignment_dim=64,
# )
# dela_history = dela_trainer.train()
# print(f"Completed {len(dela_history)} DELA-Net epoch(s).")

In [ ]:
# num_slots = int(os.environ.get("KAGGLE_TRAIN_NUM_SLOTS", "800"))
# if num_slots <= 0:
#     raise ValueError("KAGGLE_TRAIN_NUM_SLOTS must be positive.")

# recurrent_trainer = InstanSegModel(
#     data_dir=str(DATASET_ROOT),
#     dataset=str(serialized_datasets[0]) if serialized_datasets else None,
#     experiment_name="kaggle_livecell_affinity_guided_recurrent",
#     model_name="affinity_guided_recurrent",
#     imgsz=imgsz,
#     train_data_ratio=train_data_ratio,
#     seed=42,
#     dim_in=1,
#     epochs=epochs,
#     patience=2,
#     batch_size=1,
#     val_ratio=0.2,
#     test_ratio=0.2,
#     val_interval=1,
#     warmup_epochs=0,
#     resume=False,
#     amp=True,
#     num_workers=0,
#     input_channels=1,
#     backbone_channels=[16, 32, 64, 128, 256],
#     feature_channels=16,
#     num_slots=num_slots,
#     recurrent_hidden_channels=16,
#     recurrent_iterations=3,
#     slot_iterations=2,
#     minimum_instance_area=0,
# )
# history = recurrent_trainer.train()
# print(f"Completed {len(history)} recurrent instance-segmentation epoch(s).")

In [ ]:
# from model import InstanSegModel

# semantic_trainer = InstanSegModel(
#     data_dir=str(DATASET_ROOT),
#     dataset=str(serialized_datasets[0]) if serialized_datasets else None,
#     experiment_name="kaggle_livecell_semantic_unet",
#     imgsz=imgsz,
#     train_data_ratio=train_data_ratio,
#     seed=42,
#     dim_in=1,
#     epochs=epochs,
#     patience=2,
#     batch_size=2,
#     val_ratio=0.2,
#     test_ratio=0.2,
#     val_interval=1,
#     warmup_epochs=0,
#     resume=False,
#     amp=True,
#     num_workers=0,
#     input_channels=1,
#     model_name="semantic_unet",
#     unet_channels=[32, 64, 128, 256, 512],
# )
# semantic_history = semantic_trainer.train()
# print(f"Completed {len(semantic_history)} semantic U-Net epoch(s).")

In [ ]:
# Distance-transform watershed: one-channel U-Net foreground prediction,
# followed by EDT local maxima and seeded watershed instance reconstruction.
watershed_peak_distance = int(os.environ.get('KAGGLE_TRAIN_WATERSHED_PEAK_DISTANCE', '5'))
watershed_gaussian_sigma = float(os.environ.get('KAGGLE_TRAIN_WATERSHED_SIGMA', '0.0'))
watershed_minimum_area = int(os.environ.get('KAGGLE_TRAIN_WATERSHED_MIN_AREA', '0'))
watershed_connectivity = int(os.environ.get('KAGGLE_TRAIN_WATERSHED_CONNECTIVITY', '8'))
if watershed_peak_distance < 1:
    raise ValueError('KAGGLE_TRAIN_WATERSHED_PEAK_DISTANCE must be positive.')
if watershed_gaussian_sigma < 0:
    raise ValueError('KAGGLE_TRAIN_WATERSHED_SIGMA must be non-negative.')
if watershed_minimum_area < 0:
    raise ValueError('KAGGLE_TRAIN_WATERSHED_MIN_AREA must be non-negative.')
if watershed_connectivity not in {4, 8}:
    raise ValueError('KAGGLE_TRAIN_WATERSHED_CONNECTIVITY must be 4 or 8.')

watershed_trainer = InstanSegModel(
    data_dir=str(DATASET_ROOT),
    dataset=str(serialized_datasets[0]) if serialized_datasets else None,
    experiment_name='kaggle_livecell_distance_transform_watershed',
    model_name='distance_transform_watershed',
    imgsz=imgsz,
    train_data_ratio=train_data_ratio,
    seed=42,
    dim_in=1,
    epochs=epochs,
    patience=2,
    batch_size=2,
    val_ratio=0.2,
    test_ratio=0.2,
    val_interval=1,
    warmup_epochs=0,
    resume=False,
    amp=True,
    num_workers=0,
    input_channels=1,
    unet_channels=[32, 64, 128, 256, 512],
    foreground_threshold=0.5,
    peak_min_distance=watershed_peak_distance,
    gaussian_sigma=watershed_gaussian_sigma,
    minimum_instance_area=watershed_minimum_area,
    connectivity=watershed_connectivity,
)
watershed_history = watershed_trainer.train()
print(f"Completed {len(watershed_history)} distance-transform watershed epoch(s).")

In [ ]:
# Learned-distance watershed: predicts foreground, center distance, and boundary distance.
learned_center_threshold = float(os.environ.get("KAGGLE_TRAIN_LEARNED_CENTER_THRESHOLD", "0.35"))
learned_boundary_threshold = float(os.environ.get("KAGGLE_TRAIN_LEARNED_BOUNDARY_THRESHOLD", "0.5"))
learned_minimum_marker_area = int(os.environ.get("KAGGLE_TRAIN_LEARNED_MIN_MARKER_AREA", "1"))
learned_minimum_instance_area = int(os.environ.get("KAGGLE_TRAIN_LEARNED_MIN_INSTANCE_AREA", "0"))
learned_connectivity = int(os.environ.get("KAGGLE_TRAIN_LEARNED_CONNECTIVITY", "8"))
if learned_minimum_marker_area < 1:
    raise ValueError("KAGGLE_TRAIN_LEARNED_MIN_MARKER_AREA must be positive.")
if learned_minimum_instance_area < 0:
    raise ValueError("KAGGLE_TRAIN_LEARNED_MIN_INSTANCE_AREA must be non-negative.")
if learned_connectivity not in {4, 8}:
    raise ValueError("KAGGLE_TRAIN_LEARNED_CONNECTIVITY must be 4 or 8.")

learned_distance_trainer = InstanSegModel(
    data_dir=str(DATASET_ROOT),
    dataset=str(serialized_datasets[0]) if serialized_datasets else None,
    experiment_name="kaggle_livecell_learned_distance_watershed",
    model_name="learned_distance_watershed",
    imgsz=imgsz,
    train_data_ratio=train_data_ratio,
    seed=42,
    dim_in=1,
    epochs=epochs,
    patience=2,
    batch_size=2,
    val_ratio=0.2,
    test_ratio=0.2,
    val_interval=1,
    warmup_epochs=0,
    resume=False,
    amp=True,
    num_workers=0,
    input_channels=1,
    unet_channels=[32, 64, 128, 256, 512],
    foreground_threshold=0.5,
    center_distance_threshold=learned_center_threshold,
    boundary_distance_threshold=learned_boundary_threshold,
    minimum_marker_area=learned_minimum_marker_area,
    minimum_instance_area=learned_minimum_instance_area,
    connectivity=learned_connectivity,
)
learned_distance_history = learned_distance_trainer.train()
print(f"Completed {len(learned_distance_history)} learned-distance watershed epoch(s).")


In [ ]:
OUTPUT_ROOT = Path("/kaggle/working")
RUNS_SOURCE = REPOSITORY_DIR / "runs"
RUNS_OUTPUT = OUTPUT_ROOT / "runs"
if not RUNS_SOURCE.is_dir():
    raise FileNotFoundError(f"Training artifacts were not created: {RUNS_SOURCE}")
if RUNS_OUTPUT.exists():
    shutil.rmtree(RUNS_OUTPUT)
shutil.copytree(RUNS_SOURCE, RUNS_OUTPUT)
print(f"Artifacts copied to {RUNS_OUTPUT}")